In [ ]:
!pip install timm -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm
import pandas as pd
import numpy as np
import json
import os
import random
from PIL import Image, ImageEnhance
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    confusion_matrix, roc_auc_score,
    f1_score, accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

drive.mount('/content/drive')

BASE          = 'YOUR_DRIVE_PATH'
HUMAN_FOLDER  = f'{BASE}/images_human_fixed'
AUG_FOLDER    = f'{BASE}/images_human_aug'
RESULT_FOLDER = f'{BASE}/results'
FEATURE_PATH  = f'{BASE}/labels/vqa_features.json'
MODEL_FOLDER  = f'{BASE}/models_focal'

os.makedirs(MODEL_FOLDER,  exist_ok=True)
os.makedirs(RESULT_FOLDER, exist_ok=True)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
CUTOFF = 6

train_df = pd.read_csv(
    f'{BASE}/labels/human_train.csv'
)
val_df   = pd.read_csv(
    f'{BASE}/labels/human_val.csv'
)
test_df  = pd.read_csv(
    f'{BASE}/labels/human_test.csv'
)
for d in [train_df, val_df, test_df]:
    d['dementia_risk'] = d['dementia_risk'].astype(bool)

with open(FEATURE_PATH) as f:
    vqa_features = json.load(f)

def get_vqa_feat(filename, vqa_dict):
    if filename in vqa_dict:
        return vqa_dict[filename]
    tif = filename.replace('.jpg', '.tif')
    if tif in vqa_dict:
        return vqa_dict[tif]
    jpg = filename.replace('.tif', '.jpg')
    if jpg in vqa_dict:
        return vqa_dict[jpg]
    return {}

print(f'Device: {device}')
print(f'Train : {len(train_df)} รูป')

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss สำหรับ classification
    FL(pt) = -alpha * (1-pt)^gamma * log(pt)

    gamma=0 → CrossEntropy ปกติ
    gamma=2 → โฟกัส hard samples มากขึ้น
    """
    def __init__(self, gamma=2.0,
                 alpha=None, reduction='mean'):
        super().__init__()
        self.gamma     = gamma
        self.alpha     = alpha
        self.reduction = reduction

    def forward(self, pred, target):
        ce_loss = F.cross_entropy(
            pred, target, reduction='none'
        )
        pt   = torch.exp(-ce_loss)
        loss = (1 - pt) ** self.gamma * ce_loss

        if self.alpha is not None:
            alpha_t = self.alpha[target]
            loss    = alpha_t * loss

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


class FocalLossReg(nn.Module):
    """
    Focal Loss สำหรับ regression
    ให้ penalty มากขึ้นเมื่อ error สูง
    """
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, pred, target):
        mse  = F.mse_loss(pred, target,
                          reduction='none')
        err  = mse / 4.0
        loss = (err ** self.gamma) * mse
        return loss.mean()


def compute_alpha(train_df, domain_col, device):
    """คำนวณ class weight จาก inverse frequency"""
    counts = train_df[domain_col].value_counts()
    total  = len(train_df)
    alpha  = torch.zeros(3)
    for cls in [0, 1, 2]:
        n          = counts.get(cls, 1)
        alpha[cls] = total / (3 * n)
    alpha = alpha / alpha.sum() * 3
    return alpha.to(device)

test_pred  = torch.randn(4, 3)
test_label = torch.tensor([0, 1, 2, 1])
fl = FocalLoss(gamma=2.0)
print(f'Focal Loss test: {fl(test_pred, test_label):.4f}')

In [ ]:
class CDTDataset(Dataset):
    def __init__(self, df, image_folder,
                 aug_folder=None, transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']
        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224,224),
                           (255,255,255))
        if self.transform:
            img = self.transform(img)
        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)
        return img, labels

class CDTModel(nn.Module):
    def __init__(self, backbone_name='vit',
                 num_domains=5, num_classes=3):
        super().__init__()
        if backbone_name == 'vit':
            self.backbone = timm.create_model(
                'vit_base_patch16_224',
                pretrained=True, num_classes=0
            )
            self.feature_dim = 768
        else:
            backbone = models.resnet50(
                weights=models.ResNet50_Weights.IMAGENET1K_V1
            )
            self.feature_dim = 2048
            self.backbone = nn.Sequential(
                *list(backbone.children())[:-1]
            )
        self.backbone_name = backbone_name
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(), nn.Dropout(0.3)
        )
        self.heads_cls = nn.ModuleList([
            nn.Linear(512, num_classes)
            for _ in range(num_domains)
        ])
        self.heads_reg = nn.ModuleList([
            nn.Linear(512, 1)
            for _ in range(num_domains)
        ])

    def forward(self, x, mode='reg'):
        if self.backbone_name == 'vit':
            feat = self.backbone(x)
            feat = feat.unsqueeze(-1).unsqueeze(-1)
        else:
            feat = self.backbone(x)
        feat = self.shared(feat)
        if mode == 'reg':
            return [h(feat).squeeze(-1)
                    for h in self.heads_reg]
        return [h(feat) for h in self.heads_cls]

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

In [ ]:
DOMAIN_WEIGHTS = torch.tensor(
    [1.0, 1.5, 1.0, 1.0, 1.0]
).to(device)

domain_cols = [
    'domain_A','domain_B','domain_C',
    'domain_D','domain_E'
]
domain_alphas = [
    compute_alpha(train_df, col, device)
    for col in domain_cols
]

print('Class weights per domain:')
for col, alpha in zip(domain_cols, domain_alphas):
    print(f'  {col}: {alpha.cpu().numpy().round(2)}')

def compute_focal_loss(outputs, labels,
                        mode='cls', gamma=2.0):
    total = 0
    for i, output in enumerate(outputs):
        if mode == 'cls':
            fl   = FocalLoss(
                gamma=gamma,
                alpha=domain_alphas[i]
            )
            loss = fl(output, labels[:, i].long())
        else:
            fl   = FocalLossReg(gamma=gamma)
            loss = fl(output, labels[:, i].float())
        total += loss * DOMAIN_WEIGHTS[i]
    return total / DOMAIN_WEIGHTS.sum()

def get_predictions(outputs, mode='reg'):
    preds = []
    for output in outputs:
        if mode == 'reg':
            pred = output.round().clamp(0,2).long()
        else:
            pred = output.argmax(dim=1)
        preds.append(pred)
    return torch.stack(preds, dim=1)

def evaluate_model(model, loader, mode='reg'):
    model.eval()
    all_preds  = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(device)
            outputs = model(images, mode=mode)
            preds   = get_predictions(outputs, mode)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    pred_total    = all_preds.sum(axis=1)
    true_total    = all_labels.sum(axis=1)
    pred_dementia = (pred_total < CUTOFF).astype(int)
    true_dementia = (true_total < CUTOFF).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        true_dementia, pred_dementia, labels=[0,1]
    ).ravel()
    sens = tp/(tp+fn) if (tp+fn) > 0 else 0
    spec = tn/(tn+fp) if (tn+fp) > 0 else 0
    try:
        auc = roc_auc_score(
            true_dementia, -pred_total
        )
    except:
        auc = 0
    return {
        'sensitivity'  : sens,
        'specificity'  : spec,
        'auc'          : auc,
        'all_preds'    : all_preds,
        'all_labels'   : all_labels,
        'pred_total'   : pred_total,
        'true_dementia': true_dementia,
    }

In [ ]:
TARGET_NORMAL   = 2000
TARGET_DEMENTIA = 1200

dementia_df    = train_df[train_df['dementia_risk']]
normal_df      = train_df[~train_df['dementia_risk']]
normal_sampled = normal_df.sample(
    n=min(TARGET_NORMAL, len(normal_df)),
    random_state=42
)

need_more = max(0, TARGET_DEMENTIA - len(dementia_df))
aug_rows  = []
generated = 0

if need_more > 0:
    while generated < need_more:
        for _, row in dementia_df.iterrows():
            if generated >= need_more:
                break
            try:
                src = f'{HUMAN_FOLDER}/{row["filename"]}'
                img = Image.open(src).convert('RGB')
                factor = random.uniform(0.8, 1.2)
                img    = ImageEnhance.Brightness(
                    img
                ).enhance(factor)
                new_name = (
                    f'aug_f_{generated:05d}_'
                    f'{row["filename"]}'
                )
                img.save(
                    f'{AUG_FOLDER}/{new_name}',
                    'JPEG', quality=95
                )
                new_row             = row.copy()
                new_row['filename'] = new_name
                aug_rows.append(new_row)
                generated += 1
            except:
                continue

    aug_df      = pd.DataFrame(aug_rows)
    dementia_df = pd.concat(
        [dementia_df, aug_df], ignore_index=True
    )

train_balanced = pd.concat(
    [normal_sampled, dementia_df],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(
    drop=True
)

train_ds = CDTDataset(
    train_balanced, HUMAN_FOLDER,
    AUG_FOLDER, train_transform
)
val_ds = CDTDataset(
    val_df, HUMAN_FOLDER, None, val_transform
)
test_ds = CDTDataset(
    test_df, HUMAN_FOLDER, None, val_transform
)

train_loader = DataLoader(
    train_ds, batch_size=16,
    shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_ds, batch_size=16,
    shuffle=False, num_workers=2
)
test_loader = DataLoader(
    test_ds, batch_size=16,
    shuffle=False, num_workers=2
)

print(f'Train: {len(train_ds)} รูป')
print(f'Val  : {len(val_ds)} รูป')
print(f'Test : {len(test_ds)} รูป')

In [ ]:
def train_focal(backbone, mode, model_name,
                gamma=2.0, epochs=20,
                patience=10):
    print(f'\n{"="*60}')
    print(f'Training: {model_name}')
    print(f'Focal Loss gamma={gamma} mode={mode}')
    print(f'{"="*60}')

    model     = CDTModel(backbone_name=backbone).to(device)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad,
               model.parameters()),
        lr=1e-3, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )

    best_combined = -1
    best_state    = None
    no_improve    = 0
    history       = {
        'train_loss': [], 'val_loss': [],
        'sensitivity': [], 'specificity': [],
        'auc': []
    }

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(images, mode=mode)
            loss    = compute_focal_loss(
                outputs, labels, mode, gamma
            )
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        val_results = evaluate_model(
            model, val_loader, mode=mode
        )

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images, mode=mode)
                loss    = compute_focal_loss(
                    outputs, labels, mode, gamma
                )
                val_loss += loss.item()
        val_loss /= len(val_loader)

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['sensitivity'].append(
            val_results['sensitivity'])
        history['specificity'].append(
            val_results['specificity'])
        history['auc'].append(val_results['auc'])

        combined = (
            val_results['sensitivity'] +
            val_results['specificity']
        ) / 2

        if combined > best_combined:
            best_combined = combined
            best_state    = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stop epoch {epoch+1}')
                break

        print(f'Epoch {epoch+1:2d}/{epochs} | '
              f'Loss: {train_loss:.4f}/{val_loss:.4f} | '
              f'Sens: {val_results["sensitivity"]:.3f} | '
              f'Spec: {val_results["specificity"]:.3f} | '
              f'AUC: {val_results["auc"]:.3f}')

    model.load_state_dict(best_state)
    torch.save(
        best_state,
        f'{MODEL_FOLDER}/{model_name}_best.pth'
    )
    return model, evaluate_model(
        model, test_loader, mode=mode
    )

In [ ]:
results_focal = {}
for gamma in [1.0, 2.0, 3.0]:
    model_name = f'focal_vit_cls_g{int(gamma*10)}'
    path       = f'{MODEL_FOLDER}/{model_name}_best.pth'

    if os.path.exists(path):
        print(f'โหลด {model_name}')
        model = CDTModel('vit').to(device)
        model.load_state_dict(
            torch.load(path, map_location=device)
        )
        model.eval()
        test_results = evaluate_model(
            model, test_loader, 'cls'
        )
    else:
        _, test_results = train_focal(
            'vit', 'cls', model_name,
            gamma=gamma, epochs=15
        )

    results_focal[f'gamma={gamma}'] = test_results
    print(f'\nGamma={gamma}:')
    print(f'  Sens={test_results["sensitivity"]:.3f} '
          f'Spec={test_results["specificity"]:.3f} '
          f'AUC={test_results["auc"]:.3f}')

_, res_reg = train_focal(
    'vit', 'reg', 'focal_vit_reg_g20',
    gamma=2.0, epochs=15
)
results_focal['vit_reg_focal'] = res_reg

In [ ]:
all_results = {
    '07k ViT REG'      : {'sens':0.751,'spec':0.844,'auc':0.882},
    '07l Hybrid VQA'   : {'sens':0.760,'spec':0.850,'auc':0.887},
    '07n ViT CLS'      : {'sens':0.791,'spec':0.814,'auc':0.880},
}
for name, r in results_focal.items():
    all_results[f'07r Focal {name}'] = {
        'sens': r['sensitivity'],
        'spec': r['specificity'],
        'auc' : r['auc'],
    }

print('='*75)
print('COMPARISON: All Models vs Focal Loss')
print('='*75)
print(f'{"Model":<28} {"Sens":>10} '
      f'{"Spec":>10} {"AUC":>10}')
print('='*75)

best_auc = max(v['auc'] for v in all_results.values())
for name, r in all_results.items():
    mark = '✅' if r['auc'] == best_auc else ''
    goal = '🎯' if (
        r['sens'] >= 0.88 and r['spec'] >= 0.82
    ) else ''
    print(f'{name:<28} '
          f'{r["sens"]:>10.3f} '
          f'{r["spec"]:>10.3f} '
          f'{r["auc"]:>10.3f} {mark}{goal}')

print('='*75)
print(f'{"Target":<28} {"≥0.880":>10} '
      f'{"≥0.820":>10} {"≥0.910":>10}')

with open(
    f'{RESULT_FOLDER}/07r_focal_results.json', 'w'
) as f:
    json.dump({
        k: {m: float(v) for m, v in r.items()
            if isinstance(v, (int, float))}
        for k, r in all_results.items()
    }, f, indent=2)

In [ ]:
best_focal_name = 'focal_vit_cls_g20'
best_focal_path = (
    f'{BASE}/models_focal/{best_focal_name}_best.pth'
)

focal_model = CDTModel('vit').to(device)
focal_model.load_state_dict(
    torch.load(best_focal_path, map_location=device)
)
focal_model.eval()

focal_preds  = []
focal_labels = []
focal_raw    = []

with torch.no_grad():
    for images, labels in test_loader:
        images  = images.to(device)
        outputs = focal_model(images, mode='cls')

        preds = get_predictions(outputs, 'cls')
        raw   = torch.stack([
            (torch.softmax(o, dim=1) *
             torch.tensor([0.,1.,2.]).to(device)
            ).sum(dim=1)
            for o in outputs
        ], dim=1)

        focal_preds.append(preds.cpu())
        focal_labels.append(labels.cpu())
        focal_raw.append(raw.cpu())

focal_preds  = torch.cat(focal_preds).numpy()
focal_labels = torch.cat(focal_labels).numpy()
focal_raw    = torch.cat(focal_raw).numpy()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

domain_names = {
    0: 'A: digit_count',
    1: 'B: worst_quadrant',
    2: 'C: spatial',
    3: 'D: hands_present',
    4: 'E: hands_placement',
}

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

for d_idx, d_name in domain_names.items():
    true_d = focal_labels[:, d_idx].astype(int)
    pred_d = focal_preds[:, d_idx].astype(int)

    cm      = confusion_matrix(
        true_d, pred_d, labels=[0,1,2]
    )
    cm_norm = cm.astype(float)
    rs      = cm.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1
    cm_norm = cm_norm / rs

    for row_idx, (data, fmt, sfx) in enumerate([
        (cm,      'd',   ''),
        (cm_norm, '.2f', '(normalized)')
    ]):
        ax = axes[row_idx][d_idx]
        sns.heatmap(
            data, annot=True, fmt=fmt,
            cmap='Blues', ax=ax,
            xticklabels=['0','1','2'],
            yticklabels=['0','1','2'],
            cbar=False
        )
        acc = (true_d == pred_d).mean()
        ax.set_title(
            f'{d_name}\nAcc={acc:.3f} {sfx}',
            fontsize=8, fontweight='bold'
        )
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')

plt.suptitle(
    'Confusion Matrix — 07r Focal Loss (gamma=2.0)\n'
    'ViT CLS + Focal Loss',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/07r_confusion_matrix.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
from scipy.stats import gaussian_kde

colors_true = {
    0: '#3F51B5',
    1: '#00BCD4',
    2: '#8BC34A',
}

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for d_idx, d_name in domain_names.items():
    ax      = axes[d_idx]
    true_d  = focal_labels[:, d_idx].astype(int)
    score_d = focal_raw[:, d_idx]

    for true_val in [0, 1, 2]:
        mask   = (true_d == true_val)
        scores = score_d[mask]
        n      = mask.sum()
        if n < 2:
            continue
        try:
            kde    = gaussian_kde(
                scores, bw_method=0.3
            )
            x_vals = np.linspace(
                scores.min()-0.2,
                scores.max()+0.2, 300
            )
            y_vals = kde(x_vals)
            ax.plot(
                x_vals, y_vals,
                color=colors_true[true_val],
                linewidth=2,
                label=f'true={true_val} (n={n})'
            )
            ax.fill_between(
                x_vals, y_vals,
                color=colors_true[true_val],
                alpha=0.15
            )
        except:
            pass

    for x in [0.5, 1.5]:
        ax.axvline(
            x=x, color='gray',
            linestyle='--', alpha=0.4
        )
    ax.set_title(d_name, fontsize=9,
                 fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_ylabel('Density')
    if d_idx == 0:
        ax.legend(fontsize=7)

plt.suptitle(
    'Score Distribution — 07r Focal Loss (gamma=2.0)\n'
    'blue=0  cyan=1  green=2',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/07r_distribution.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

models_compare = {
    '07k ViT REG'      : {
        'sens':0.751,'spec':0.844,'auc':0.882,
        'color':'#9E9E9E'
    },
    '07l Hybrid VQA'   : {
        'sens':0.760,'spec':0.850,'auc':0.887,
        'color':'#2196F3'
    },
    '07r Focal g=1.0'  : {
        'sens':0.853,'spec':0.743,'auc':0.875,
        'color':'#FF9800'
    },
    '07r Focal g=2.0'  : {
        'sens':0.884,'spec':0.701,'auc':0.893,
        'color':'#F44336'
    },
    '07r Focal g=3.0'  : {
        'sens':0.871,'spec':0.733,'auc':0.877,
        'color':'#E91E63'
    },
}

names  = list(models_compare.keys())
colors = [v['color'] for v in models_compare.values()]

for ax_idx, (metric, title, target) in enumerate([
    ('sens', 'Sensitivity', 0.88),
    ('spec', 'Specificity', 0.82),
    ('auc',  'AUC-ROC',     0.91),
]):
    vals = [
        models_compare[n][metric] for n in names
    ]
    bars = axes[ax_idx].barh(
        names, vals, color=colors,
        edgecolor='white', height=0.6
    )
    axes[ax_idx].axvline(
        x=target, color='red',
        linestyle='--', linewidth=1.5,
        label=f'Target {target}'
    )
    axes[ax_idx].set_title(
        title, fontsize=12, fontweight='bold'
    )
    axes[ax_idx].set_xlim(0.5, 1.05)
    axes[ax_idx].legend(fontsize=8)

    for bar, val in zip(bars, vals):
        axes[ax_idx].text(
            val + 0.003,
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}',
            va='center', fontsize=8
        )

plt.suptitle(
    'Model Comparison: 07k vs 07l vs 07r\n'
    'Grey=07k | Blue=07l | Orange/Red=07r Focal',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/07r_comparison.png',
    dpi=150, bbox_inches='tight'
)
plt.show()